# Praktikum 06 – RAG-Systeme und Retrieval-Pipeline
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Mehrseitige Web-Extraktion planen, Chunking-Strategien vergleichen, eine vollständige RAG-Pipeline aufbauen und Retrieval-Qualität begründet auswerten.

**Zielprodukt nach 90 Minuten:**
1. Ein kleiner, reproduzierbarer Korpus aus mehreren Wikipedia-Artikeln.
2. Ein Vergleich von zwei Chunking-Strategien mit denselben Suchanfragen.
3. Ein ChromaDB-Vektorspeicher mit den gecrawlten Dokumenten.
4. Eine vollständige RAG-Pipeline mit Retrieval und Generation.
5. Ein Vergleich von Vektor- und Hybridsuche.

**Arbeitsmodus:** Die Pflichtteile laufen von oben nach unten. Die Aufgaben am Ende sind die Abgabe.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import tempfile
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "chromadb": ("chromadb", "1.1.1"),
    "requests": ("requests", "2.33.1"),
    "bs4": ("beautifulsoup4", "4.14.3"),
    "ollama": ("ollama", "0.6.1"),
    "numpy": ("numpy", "2.4.4"),
    "sklearn": ("scikit-learn", "1.8.0"),
    "matplotlib": ("matplotlib", "3.10.8"),
    "pandas": ("pandas", "3.0.2"),
    "tiktoken": ("tiktoken", "0.12.0"),
}



def run_install(specs):
    commands = []

    if shutil.which("uv") is not None:
        commands.append(["uv", "pip", "install", "--python", sys.executable, *specs])

    commands.append([sys.executable, "-m", "pip", "install", *specs])
    if not IN_COLAB:
        commands.append([sys.executable, "-m", "pip", "install", "--user", *specs])

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    install_name
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import chromadb
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import numpy as np
import ollama
import tiktoken

DEFAULT_OPTIONS = {
    "num_ctx": 8192,
    "num_predict": 512,
    "temperature": 0.5,
}


def build_options(**overrides):
    options = DEFAULT_OPTIONS.copy()
    for key, value in overrides.items():
        if value is not None:
            options[key] = value
    return options


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


def model_is_available(requested_name, available_names):
    candidates = {requested_name, f"{requested_name}:latest"}
    return any(candidate in available_names for candidate in candidates)


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if sys.platform.startswith("win"):
        raise RuntimeError("Ollama fehlt unter Windows. Installiere es über https://ollama.com/download/windows und führe die Setup-Zelle erneut aus.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    log_path = os.path.join(tempfile.gettempdir(), "ollama-notebook.log")
    ollama_log = open(log_path, "a", encoding="utf-8")
    popen_kwargs = {"stdout": ollama_log, "stderr": subprocess.STDOUT}
    if os.name != "nt":
        popen_kwargs["start_new_session"] = True
    subprocess.Popen(["ollama", "serve"], **popen_kwargs)
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
for model_name in [MODEL, EMBED_MODEL]:
    subprocess.check_call(["ollama", "pull", model_name], env=env)

payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
missing_models = [
    model_name
    for model_name in [MODEL, EMBED_MODEL]
    if not model_is_available(model_name, available_models)
]
if missing_models:
    raise RuntimeError(f"Missing local Ollama models: {', '.join(missing_models)}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Models:", MODEL, ",", EMBED_MODEL)
print("Available local models:", ", ".join(available_models))


## Teil 1 – Mehrseitiges Scraping und Korpusaufbau (20 min)
Wir bauen einen kleinen Korpus aus mehreren thematisch verwandten Wikipedia-Artikeln auf.

**Pflichtschritte:**
- Starte mit dem KI-Artikel und folge internen Wikipedia-Links bis maximal 4 Seiten.
- Extrahiere pro Seite nur inhaltliche Abschnitte mit ausreichend Text.
- Prüfe danach, aus welchen Seiten und Abschnitten dein Korpus besteht.

**Soll-Ergebnis:** eine Tabelle mit Seiten, Abschnittstiteln und Textlängen.

In [ ]:
import re
from collections import deque
from urllib.parse import urldefrag, urljoin, urlparse

import pandas as pd

START_URL = "https://de.wikipedia.org/wiki/K%C3%BCnstliche_Intelligenz"
ARTICLE_PREFIX = "https://de.wikipedia.org/wiki/"
MAX_PAGES = 4
MAX_LINKS_PER_PAGE = 8
MIN_SECTION_CHARS = 180
USER_AGENT = {"User-Agent": "Mozilla/5.0 (Praktikum AGI NLP)"}
PRIORITY_LINK_HINTS = ["Maschinelles_Lernen", "Neuronales_Netz", "Deep_Learning"]


def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_section_title(text):
    text = normalize_text(text)
    text = re.sub(r"\[\s*Bearbeiten\s*\|\s*Quelltext bearbeiten\s*\]$", "", text)
    return normalize_text(text)


def is_allowed_article(url):
    cleaned_url, _ = urldefrag(url)
    parsed = urlparse(cleaned_url)
    if parsed.scheme not in {"http", "https"}:
        return False
    article_name = cleaned_url.removeprefix(ARTICLE_PREFIX)
    return cleaned_url.startswith(ARTICLE_PREFIX) and ":" not in article_name


def link_priority(url):
    lowered_url = url.lower()
    for index, hint in enumerate(PRIORITY_LINK_HINTS):
        if hint.lower() in lowered_url:
            return index
    return len(PRIORITY_LINK_HINTS)


def extract_links(soup, base_url):
    links = []
    seen = set()
    for tag in soup.select(".mw-parser-output a[href]"):
        candidate_url = urljoin(base_url, tag["href"])
        candidate_url, _ = urldefrag(candidate_url)
        if not is_allowed_article(candidate_url):
            continue
        if candidate_url in seen:
            continue
        seen.add(candidate_url)
        links.append(candidate_url)
    links.sort(key=lambda url: (link_priority(url), url))
    return links


def extract_section_rows(section, page_title, page_url, rows):
    heading_container = next(
        (
            child
            for child in section.children
            if getattr(child, "name", None) == "div" and "mw-heading" in (child.get("class") or [])
        ),
        None,
    )

    section_title = "Einleitung"
    if heading_container is not None:
        heading = heading_container.find(re.compile(r"^h[1-6]$"), recursive=False)
        if heading is not None:
            section_title = clean_section_title(heading.get_text(" ", strip=True))

    text_parts = []
    for child in section.children:
        if not getattr(child, "name", None):
            continue
        if child == heading_container or child.name == "section":
            continue
        if child.name == "div" and "hauptartikel" in (child.get("class") or []):
            continue
        if child.name in {"p", "ul", "ol", "blockquote", "dl"}:
            block_text = normalize_text(child.get_text(" ", strip=True))
            if block_text:
                text_parts.append(block_text)

    text = " ".join(text_parts)
    if len(text) >= MIN_SECTION_CHARS:
        rows.append(
            {
                "page_title": page_title,
                "page_url": page_url,
                "section_title": section_title,
                "content": text,
                "char_count": len(text),
            }
        )

    for nested_section in section.find_all("section", recursive=False):
        extract_section_rows(nested_section, page_title, page_url, rows)


def crawl_articles(start_url, max_pages=MAX_PAGES):
    queue = deque([start_url])
    visited_urls = []
    visited_set = set()
    rows = []

    while queue and len(visited_urls) < max_pages:
        current_url = queue.popleft()
        if current_url in visited_set:
            continue

        response = requests.get(current_url, headers=USER_AGENT, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        content_root = soup.find(class_="mw-parser-output")
        if content_root is None:
            raise RuntimeError(f"Could not find article content for {current_url}")

        title_tag = soup.find("h1")
        page_title = normalize_text(title_tag.get_text(" ", strip=True)) if title_tag else current_url.rsplit("/", 1)[-1]

        top_sections = content_root.find_all("section", recursive=False)
        if not top_sections:
            raise RuntimeError(f"No top-level sections found for {current_url}")

        for top_section in top_sections:
            extract_section_rows(top_section, page_title, current_url, rows)

        visited_urls.append(current_url)
        visited_set.add(current_url)

        for link in extract_links(soup, current_url):
            if link in visited_set or link in queue:
                continue
            queue.append(link)
            if len(queue) >= MAX_LINKS_PER_PAGE:
                break

    if len(visited_urls) < max_pages:
        raise RuntimeError(f"Only crawled {len(visited_urls)} pages although {max_pages} were requested.")
    if not rows:
        raise RuntimeError("The crawl did not yield any suitable text sections.")

    return pd.DataFrame(rows), visited_urls


sections_df, crawled_urls = crawl_articles(START_URL)
sections_df = sections_df.sort_values(["page_title", "section_title"]).reset_index(drop=True)

print("Gecrawlte Seiten:")
for url in crawled_urls:
    print(f"- {url}")

print(f"\nExtrahierte Abschnitte: {len(sections_df)}")
print(sections_df[["page_title", "section_title", "char_count"]].head(12).to_string(index=False))

In [ ]:
page_summary = (
    sections_df.groupby("page_title")
    .agg(anzahl_abschnitte=("content", "count"), mittlere_zeichen=("char_count", "mean"))
    .reset_index()
    .sort_values("anzahl_abschnitte", ascending=False)
    .reset_index(drop=True)
)
page_summary["mittlere_zeichen"] = page_summary["mittlere_zeichen"].round(0).astype(int)

print(page_summary.to_string(index=False))

example_row = sections_df.iloc[0]
print("\nBeispielabschnitt:")
print(f"{example_row['page_title']} | {example_row['section_title']}")
print(example_row["content"][:600] + "...")

## Teil 2 – Chunking-Strategien vergleichen (20 min)
Wir vergleichen ein tokenbasiertes Sliding Window mit einem einfachen Zeichenfenster.

**Pflichtschritte:**
- Zerlege denselben Korpus mit zwei unterschiedlichen Strategien.
- Vergleiche Anzahl, Größe und mittlere Tokenzahl der Chunks.
- Nutze danach identische Suchanfragen für beide Varianten.

**Soll-Ergebnis:** eine kleine Vergleichstabelle.

In [ ]:
ENC = tiktoken.get_encoding("cl100k_base")


def token_chunker(text, max_tokens=140, overlap=30):
    step = max_tokens - overlap
    if step <= 0:
        raise RuntimeError("Token chunking requires overlap < max_tokens.")
    tokens = ENC.encode(text)
    return [ENC.decode(tokens[start : start + max_tokens]) for start in range(0, len(tokens), step)]


def char_chunker(text, chunk_size=420, overlap=80):
    step = chunk_size - overlap
    if step <= 0:
        raise RuntimeError("Character chunking requires overlap < chunk_size.")
    return [text[start : start + chunk_size] for start in range(0, len(text), step)]


def build_chunk_table(df, strategy_name, chunk_fn):
    rows = []
    for record in df.to_dict("records"):
        chunks = chunk_fn(record["content"])
        for chunk_index, chunk in enumerate(chunks, start=1):
            cleaned_chunk = normalize_text(chunk)
            if len(cleaned_chunk) < 80:
                continue
            rows.append(
                {
                    "strategy": strategy_name,
                    "page_title": record["page_title"],
                    "section_title": record["section_title"],
                    "chunk_index": chunk_index,
                    "text": cleaned_chunk,
                    "char_count": len(cleaned_chunk),
                    "token_count": len(ENC.encode(cleaned_chunk)),
                }
            )
    if not rows:
        raise RuntimeError(f"Chunking strategy {strategy_name} produced no usable chunks.")
    return pd.DataFrame(rows)


token_chunks = build_chunk_table(sections_df, "token_sliding", token_chunker)
char_chunks = build_chunk_table(sections_df, "char_window", char_chunker)

comparison = pd.DataFrame(
    [
        {
            "strategie": "token_sliding",
            "chunks": len(token_chunks),
            "mittlere_zeichen": int(token_chunks["char_count"].mean()),
            "mittlere_tokens": int(token_chunks["token_count"].mean()),
        },
        {
            "strategie": "char_window",
            "chunks": len(char_chunks),
            "mittlere_zeichen": int(char_chunks["char_count"].mean()),
            "mittlere_tokens": int(char_chunks["token_count"].mean()),
        },
    ]
)

print(comparison.to_string(index=False))

## Teil 3 – Vektorspeicher mit ChromaDB erstellen (15 min)
Wir indexieren die gecrawlten Dokumente in einer Vektordatenbank für effizientes Retrieval.

**Pflichtschritte:**
- Wähle eine Chunking-Strategie aus (token_basiert oderzeichenbasiert).
- Erstelle eine ChromaDB-Collection und füge alle Chunks mit Embeddings hinzu.
- Prüfe, ob die erwartete Anzahl von Dokumenten in der Datenbank liegt.

**Soll-Ergebnis:** ein funktionsfähiger Vektorspeicher mit Metadaten.

In [ ]:
chosen_chunks = token_chunks
print(f"Verwende Chunking-Strategie: token_sliding mit {len(chosen_chunks)} Chunks")


def embed_text(text):
    return ollama.embeddings(model=EMBED_MODEL, prompt=text)["embedding"]


client = chromadb.PersistentClient(path="./rag_storage")
collection_name = "wiki_ai_corpus"
existing_collections = {item.name for item in client.list_collections()}
if collection_name in existing_collections:
    client.delete_collection(collection_name)
collection = client.create_collection(name=collection_name)

doc_ids = []
documents = []
embeddings = []
metadatas = []

for idx, row in chosen_chunks.reset_index(drop=True).iterrows():
    doc_id = f"doc_{idx}"
    doc_ids.append(doc_id)
    documents.append(row["text"])
    embeddings.append(embed_text(row["text"]))
    metadatas.append({
        "page_title": row["page_title"],
        "section_title": row["section_title"],
        "chunk_index": row["chunk_index"],
    })

collection.add(
    ids=doc_ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
)

topic_counts = {}
for row in chosen_chunks.to_dict("records"):
    topic_counts[row["page_title"]] = topic_counts.get(row["page_title"], 0) + 1

print(f"\nAnzahl Dokumente in der Datenbank: {collection.count()}")
print("Verteilung nach Seiten:")
for topic, count in sorted(topic_counts.items()):
    print(f"- {topic}: {count}")

## Teil 4 – Retrieval und Generation (RAG-Pipeline) (20 min)
Wir implementieren eine vollständige RAG-Pipeline mit Retrieval, Kontextaufbau und Answer Generation.

**Pflichtschritte:**
- Implementiere eine Retrieval-Funktion mit ChromaDB.
- Erstelle eine Prompt-Vorlage für die RAG-Generation.
- Beantworte Beispielfragen und dokumentiere die verwendeten Quellen.

**Soll-Ergebnis:** Eine funktionierende RAG-Pipeline mit Quelldokumentation.

In [ ]:
def vector_search(query, top_k=3, where=None):
    request = {
        "query_embeddings": [embed_text(query)],
        "n_results": min(top_k, len(chosen_chunks)),
        "include": ["documents", "metadatas", "distances"],
    }
    if where is not None:
        request["where"] = where
    result = collection.query(**request)
    return [
        {
            "document": document,
            "metadata": metadata,
            "distance": distance,
        }
        for document, metadata, distance in zip(
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0],
        )
    ]


def build_context(query, top_k=3, where=None):
    items = vector_search(query, top_k=top_k, where=where)
    blocks = [f"[{item['metadata']['page_title']} - {item['metadata']['section_title']}] {item['document']}" for item in items]
    return items, "\n\n".join(blocks)


SYSTEM_PROMPT = (
    "Du bist ein hilfreicher Assistent, der Fragen nur basierend auf dem bereitgestellten Kontext beantwortet. "
    "Antworte ausschliesslich mit Informationen aus dem Kontext. "
    "Wenn die Information im Kontext nicht vorhanden ist, antworte exakt mit 'Unbekannt'. "
    "Nenne am Ende deiner Antwort die verwendeten Quellen in Klammern."
)


def rag_answer(query, top_k=3, where=None):
    items, context = build_context(query, top_k=top_k, where=where)
    prompt = (
        f"Kontext:\n{context}\n\nFrage: {query}"
    )
    answer = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        think=False,
        options=build_options(temperature=0, num_predict=150),
        keep_alive="10m",
    )["message"]["content"].strip()
    if not answer:
        raise RuntimeError("Das Modell hat keine Antwort erzeugt.")

    return {
        "answer": answer,
        "doc_ids": [f"{item['metadata']['page_title']} - {item['metadata']['section_title']}" for item in items],
    }


demo_queries = [
    "Was ist maschinelles Lernen?",
    "Wie funktionieren neuronale Netze?",
    "Welche Rolle spielt Deep Learning?",
]

for query in demo_queries:
    report = rag_answer(query)
    print(f"Frage: {query}")
    print(f"Verwendete Quellen: {', '.join(report['doc_ids'])}")
    print(f"Antwort: {report['answer']}\n")

In [ ]:
def faithfulness_check(query, answer, context):
    check_prompt = (
        "Prüfe, ob die Antwort vollständig durch den Kontext gedeckt ist. "
        "Antorte exakt mit JA oder NEIN.\n\n"
        f"Kontext:\n{context}\n\nAntwort: {answer}"
    )
    validity = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "user", "content": check_prompt},
        ],
        think=False,
        options=build_options(temperature=0, num_predict=8, stop=["\n"]),
        keep_alive="10m",
    )["message"]["content"].strip()
    return validity


print("Faithfulness-Check für die Demo-Fragen:\n")
for query in demo_queries:
    items, context = build_context(query, top_k=3)
    report = rag_answer(query)
    validity = faithfulness_check(query, report["answer"], context)
    print(f"Frage: {query}")
    print(f"Faithfulness: {validity}\n")

## Teil 5 – Vergleich Retrieval-Methoden (15 min)
Wir vergleichen reine Vektorsuche mit einer hybriden Suche, die lexikale und semantische Signale kombiniert.

**Pflichtschritte:**
- Implementiere eine einfache hybride Suchstrategie.
- Vergleiche die Ergebnisse beider Methoden für dieselben Fragen.
- Dokumentiere, welche Methode für welche Art von Fragen besser geeignet ist.

**Soll-Ergebnis:** eine Benchmark-Tabelle für beide Retrieval-Methoden.

In [ ]:
import re

STOPWORDS = {"was", "ist", "der", "die", "das", "ein", "eine", "und", "mit", "für", "bei", "auf", "wie", "welche", "rolle"}


def lexical_score(query, document):
    query_terms = [token for token in re.findall(r"\w+", query.lower()) if token not in STOPWORDS]
    document_terms = set(re.findall(r"\w+", document.lower()))
    return sum(1 for token in query_terms if token in document_terms)


def hybrid_search(query, top_k=3, where=None):
    candidates = vector_search(query, top_k=min(8, len(chosen_chunks)), where=where)
    ranked = []
    for rank, item in enumerate(candidates, start=1):
        lex_score = lexical_score(query, item["document"])
        vector_bonus = max(0, 8 - rank)
        ranked.append({**item, "score": lex_score * 10 + vector_bonus})
    ranked.sort(key=lambda item: item["score"], reverse=True)
    return ranked[:top_k]


RETRIEVAL_BENCHMARK = [
    {"query": "Was ist maschinelles Lernen?", "expected_pages": {"Maschinelles Lernen", "Künstliche Intelligenz"}},
    {"query": "Wie funktionieren neuronale Netze?", "expected_pages": {"Neuronales Netz", "Künstliche Intelligenz"}},
    {"query": "Welche Rolle spielt Deep Learning?", "expected_pages": {"Deep Learning", "Neuronales Netz"}},
]


def evaluate_search(search_fn, label):
    rows = []
    for case in RETRIEVAL_BENCHMARK:
        results = search_fn(case["query"], top_k=3)
        retrieved_pages = [item["metadata"]["page_title"] for item in results]
        rows.append(
            {
                "query": case["query"],
                "expected": ", ".join(sorted(case["expected_pages"])),
                "top3": ", ".join(retrieved_pages),
                "treffer": "JA" if any(page in case["expected_pages"] for page in retrieved_pages) else "NEIN",
            }
        )
    print(f"\n{label}")
    for row in rows:
        print(f"- {row['treffer']} | {row['expected'][:30]}... | {row['top3'][:40]}...")
    hit_rate = sum(row["treffer"] == "JA" for row in rows) / len(rows)
    print(f"Trefferquote Top-3: {hit_rate:.2f}")
    return rows


print("Vektorsuche (rein)")
vector_rows = evaluate_search(vector_search, "Vektorsuche")

print("\nHybridsuche (Vektor + Lexikal)")
hybrid_rows = evaluate_search(hybrid_search, "Hybridsuche")

## Abgabe und Reflexion
**Pflichtabgabe:**
1. Notiere die Zahl der gecrawlten Seiten und der extrahierten Abschnitte.
2. Gib die Trefferquote für Vektorsuche und Hybridsuche an.
3. Begründe in 3 bis 5 Sätzen, welche Retrieval-Methode für den gewählten Korpus geeigneter ist.
4. Dokumentiere die Faithfulness-Ergebnisse für die RAG-Antworten.

**Transferaufgaben:**
1. Erhöhe die Crawling-Tiefe auf 5 Seiten und erstelle einen neuen Vektorspeicher.
2. Implementiere ein Re-Ranking, das kürzere Chunks bei gleichem Score bevorzugt.
3. Formuliere eine eigene Frage, die mit dem RAG-System nicht beantwortet werden kann, und prüfe, ob es mit "Unbekannt" antwortet.
4. Füge einen Metadaten-Filter hinzu (z.B. nur Seiten einer bestimmten Kategorie) und teste die Filterung.